<h1 style="text-align: left;"><strong>电商平台首页改版 A/B Test 增长实验分析</strong></h1><h1></h1><p></p>

<h2>项目背景</h2><p>某电商平台计划对首页进行改版，产品团队认为新版首页通过优化商品推荐布局、视觉设计及内容展示方式，能够提升用户点击率（CTR）和购买转化率（CVR）。</p><p>为了验证改版效果，采用A/B Test实验方法进行效果评估。</p><p></p>

<h2>数据来源</h2><p>本实验数据为模拟生成，目的是复现A/B Test分析流程。</p><p>- 样本量：200,000用户</p><p>- 用户类型：新用户70%，老用户30%</p><p>- 实验组：A组（旧版首页）、B组（新版首页），随机分配50%/50%</p><p>- 包含字段：user_id, user_type, group, device, channel, click, order, revenue, event_date</p><p>- 数据生成逻辑遵循业务假设，新用户和高意图用户在新版首页上的CTR/CVR提升更明显</p><p></p>

<h2>项目目标</h2><p>评估新版首页（B组）相较于旧版首页（A组）是否能够：</p><ul><li><p>提升用户点击率（CTR）</p></li><li><p>提升用户购买转化率（CVR）</p></li><li><p>提升整体商业价值</p></li><li><p>明确不同用户群体的响应差异</p></li></ul><p></p>

<h2>技术栈</h2><p>Python</p><p>Pandas</p><p>NumPy</p><p>Matplotlib</p><p>Seaborn</p><p>Statsmodels</p><p></p>

<h2>1.导入数据并查看</h2><p></p>

In [1]:
#导入相关库
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

#导入统计检验库
from statsmodels.stats.proportion import proportions_ztest

#设置中文字体
import matplotlib
matplotlib.rc("font",family='Microsoft YaHei')

In [2]:
#导入数据
df = pd.read_csv('abtest_ecommerce.csv')
print("数据加载成功")

#查看数据
print("数据前五条记录：")
df.head()

print("数据描述性信息：")
print(df.info())

print("数据统计性信息")
print(df.describe())

数据加载成功
数据前五条记录：
数据描述性信息：
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 200000 entries, 0 to 199999
Data columns (total 9 columns):
user_id       200000 non-null int64
group         200000 non-null object
user_type     200000 non-null object
device        200000 non-null object
channel       200000 non-null object
click         200000 non-null int64
order         200000 non-null int64
revenue       200000 non-null float64
event_date    200000 non-null object
dtypes: float64(1), int64(3), object(5)
memory usage: 13.7+ MB
None
数据统计性信息
             user_id          click          order        revenue
count  200000.000000  200000.000000  200000.000000  200000.000000
mean   100000.500000       0.089290       0.004755       0.477267
std     57735.171256       0.285163       0.068793       7.585467
min         1.000000       0.000000       0.000000       0.000000
25%     50000.750000       0.000000       0.000000       0.000000
50%    100000.500000       0.000000       0.000000       0.000

<p>针对revenue字段25%，50%，75%的分位数都是0：因为绝大多数用户在实验周期内没有下单，只有一小部分用户产生订单金额，所以 GMV 数据呈现典型长尾分布，中位数和低分位数为0，而少数高价值用户拉高了最大值。</p>

<h2>2.数据清洗</h2><p></p>

In [3]:
#检验缺失值
print(f"缺失值的个数是：{df.isnull().sum()}")

#检验重复值
print(f"重复值的个数是：{df.duplicated().sum()}")

print("实验数据整体质量较高，未发现明显缺失值与重复数据，满足后续实验分析要求。")

缺失值的个数是：user_id       0
group         0
user_type     0
device        0
channel       0
click         0
order         0
revenue       0
event_date    0
dtype: int64
重复值的个数是：0
实验数据整体质量较高，未发现明显缺失值与重复数据，满足后续实验分析要求。


<h2>3.KPI总览</h2><p></p>

In [4]:
#构建行为表
summary = df.groupby('group').agg({'click':'sum','order':'sum','user_id':'count','revenue':'sum'}).reset_index()

#计算CTR 点击率（用于衡量首页吸引用户点击的能力）
summary['CTR'] = (summary['click']/summary['user_id'])
    
 #计算CVR 转化率（用于衡量用户从点击到购买的转化能力）
summary['CVR'] = (summary['order']/summary['click'])
    
print(f"用户分布详情：{summary}")   
    
#CTR对比可视化
plt.figure(figsize=(6,4))
ax = sns.barplot(data=summary,x='group',y='CTR')
for i,v in enumerate(summary['CTR']):
    ax.text(               #在图上写字
        i,                 #x轴坐标
        v+0.001,           #y轴坐标   +0.001：让文字显示在柱子顶部稍微往上一点
        f'{v:.2%}',        #显示内容
        ha='center'        #居中
    )
plt.title('CTR Comparison')
plt.show()   

#CVR对比可视化
plt.figure(figsize=(6,4))
ax = sns.barplot(data=summary,x='group', y='CVR')
for i,v in enumerate(summary['CVR']):
    ax.text(
        i,
        v+0.001,
        f'{v:.2%}',
        ha='center'
    )
plt.title('CVR Comparison')
plt.show()

用户分布详情：  group  click  order  user_id       revenue       CTR       CVR
0     A   8081    369    99768  36847.519489  0.080998  0.045663
1     B   9777    582   100232  58605.948691  0.097544  0.059527


<Figure size 432x288 with 1 Axes>

<Figure size 432x288 with 1 Axes>

<p>从实验结果来看，B组CTR高于A组，说明新版首页能够更有效吸引用户点击。 同时： B组CVR也有所提升，说明新版首页不仅提升了用户兴趣，也改善了用户购买转化能力。</p>

<h2>4.统计性检验</h2><p></p>

<p>CTR检验</p><p></p><p>原假设H0：新版首页与旧版首页CTR没有显著差异即P(A) = P(B)</p><p></p><p>备择假设H1: 新版首页CTR显著高于旧版首页CTR即P(B) &gt; P(A)    </p><p>#我们的目标是验证新版首页是否提升CTR，因此备择假设选择 H1: P(B) &gt; P(A)。原假设 H₀ 表示两组CTR没有显著差异。选择单尾检验方向是基于业务目标：我们关心的是正向提升，而不是下降。</p>

In [5]:
#点击人数
clicks = summary['click']

#总曝光人数
impressions = summary['user_id']

#CTR提升率
ctr_lift = (summary.loc[1,'CTR']-summary.loc[0,'CTR']) / summary.loc[0,'CTR']  #（B组CTR-A组CTR）/A组CTR
    
 #执行Z检验
z_stat, p_value = proportions_ztest(count=clicks,nobs=impressions,alternative='two-sided')
    
print(f"CTR Lift: {ctr_lift:.2%}")
print("Z-statistic:", z_stat)
print("P-value:", p_value)   

#判断结果
alpha = 0.05
if p_value < alpha:
    print("拒绝原假设")
else:
    print("无法拒绝原假设")   
   
    

CTR Lift: 20.43%
Z-statistic: -12.97417362175523
P-value: 1.7143532044711046e-38
拒绝原假设


<p>检验结果显示：</p><p style="text-align: left;">新版首页CTR较旧版首页提升20.43%，且P-value小于0.05，差异具有统计显著性。</p><p style="text-align: left;">说明新版页面设计能够更有效吸引用户点击商品。</p><p style="text-align: left;">因此从用户兴趣激发角度来看，新版首页具备上线价值</p>

<p>CVR检验</p><p></p><p>原假设H0：新版首页与旧版首页CVR没有显著差异即P(A) = P(B)</p><p></p><p>备择假设H1: 新版首页CTR显著高于旧版首页CVR即P(B) &gt; P(A)</p>

In [6]:
#购买人数
orders = summary['order']

#点击人数
clicks = summary['click']

#CVR提升率
cvr_lift = (summary.loc[1,'CVR']-summary.loc[0,'CVR']) / summary.loc[0,'CVR']
    
#Z检验
z_stat_cvr, p_value_cvr = proportions_ztest(count=orders,nobs=clicks)
    
print(f"CVR Lift: {cvr_lift:.2%}")

print("Z-statistic:", z_stat_cvr)
print("P-value:", p_value_cvr)    
   
#判断结果
alpha = 0.05

if p_value < alpha:
    print("拒绝原假设")
else:
    print("无法拒绝原假设")    


CVR Lift: 30.36%
Z-statistic: -4.1071611909289345
P-value: 4.005517068969857e-05
拒绝原假设


<p>检验结果显示： 新版首页CVR较旧版首页提升30.36%，且差异具P-value小于0.05，有统计显著性。</p><p style="text-align: left;">说明新版页面设计能够更有效促使用户购买击商品。</p><p style="text-align: left;">因此从用户购买商品角度来看，新版首页具备上线价值。</p>

<h2>5.用户分群分析</h2><p></p>

In [7]:
#新老用户分群
user_summary = (df.groupby(['user_type','group']).agg({'user_id':'count','click':'sum','order':'sum','revenue':'sum'}).reset_index())
    
#字段命名                        
user_summary.columns = ['user_type','group','users','clicks','orders','revenue']


#计算CTR
user_summary['CTR'] = ( user_summary['clicks']/user_summary['users'])
   
#计算CVR
user_summary['CVR'] = (user_summary['orders']/user_summary['clicks'])

#查看新老用户信息表
print(user_summary)  
                  
#新老用户CTR对比图         
plt.figure(figsize=(8,5))
sns.barplot(data=user_summary,x='user_type',y='CTR',hue='group')
plt.title('CTR by User Type')
plt.show()
      
#新老用户CVR对比图
plt.figure(figsize=(8,5))
sns.barplot(data=user_summary,x='user_type', y='CVR',hue='group')
plt.title('CVR by User Type')
plt.show()      

  user_type group  users  clicks  orders       revenue       CTR       CVR
0       new     A  69768    5649     241  24736.789935  0.080968  0.042662
1       new     B  70347    7055     419  41957.090880  0.100289  0.059391
2       old     A  30000    2432     128  12110.729554  0.081067  0.052632
3       old     B  29885    2722     163  16648.857812  0.091082  0.059882


<Figure size 576x360 with 1 Axes>

<Figure size 576x360 with 1 Axes>

<p style="text-align: left;">新版首页对新用户提升效果最明显，CTR提升幅度显著高于老用户，说明新版首页优化的信息展示方式能够帮助首次访问用户更快发现感兴趣商品。</p><p style="text-align: left;"></p><p style="text-align: left;">新版首页同时提升了新老用户购买转化率，其中新用户CVR提升幅度更大，说明新版首页不仅提高用户点击意愿，也改善了新用户购物决策效率。</p><p></p><p>建议：优先向新用户全量推广新版首页；老用户采用个性化推荐</p><p></p>

In [8]:
#渠道分群
channel_summary = (df.groupby(['channel','group']).agg({'user_id':'count','click':'sum','order':'sum'}) .reset_index())
channel_summary.columns = ['channel','group','users','clicks','orders']
   
#计算CTR
channel_summary['CTR'] = (channel_summary['clicks']/channel_summary['users'])

#计算CVR
channel_summary['CVR'] = (channel_summary['orders']/channel_summary['clicks'])
    
#查看渠道信息表
print(channel_summary.head())    
    
#不同渠道CTR对比图
plt.figure(figsize=(10,5))
sns.barplot(data=channel_summary,x='channel',y='CTR',hue='group')
plt.title('CTR by Channel')
plt.show()
    
#不同渠道CVR对比图
plt.figure(figsize=(10,5))
sns.barplot(data=channel_summary,x='channel',y='CVR',hue='group')
plt.title('CVR by Channel')
plt.show()   

  channel group  users  clicks  orders       CTR       CVR
0     ads     A  34929    2863     136  0.081966  0.047503
1     ads     B  35079    3395     191  0.096782  0.056259
2  direct     A  19871    1624      59  0.081727  0.036330
3  direct     B  20102    1950     142  0.097005  0.072821
4  search     A  24962    2008      88  0.080442  0.043825


<Figure size 720x360 with 1 Axes>

<Figure size 720x360 with 1 Axes>

<p>新界面所有渠道的CTR和CVR都高于旧界面，说明新界面再吸引用户点击和购买方面更有效。</p><p></p><p>direct渠道的CVR提升最明显，说明新界面对已有明确意图的用户特别有效。</p><p></p><p>social渠道的CVR提升最小，说明新界面对社交渠道用户的改善转化有限，</p><p></p><p>建议：逐步上线新版界面；复盘direct渠道为什么提升最大，然后把检验迁移到其他渠道；social用户意图较弱，建议测试更情绪化的文案或更匹配广告素材的落地页。</p><p></p>

In [9]:
#设备分群
device_summary = (df.groupby(['device','group']).agg({'user_id':'count','click':'sum','order':'sum'}) .reset_index())
device_summary.columns = ['device','group','users','clicks','orders']

#计算CTR
device_summary['CTR'] = (device_summary['clicks']/device_summary['users'])
    
#计算CVR    
device_summary['CVR'] = (device_summary['orders']/device_summary['clicks'])
    
#查看设备信息表
print(device_summary.head())     
    
#不同设备CTR对比图
plt.figure(figsize=(10,5))
sns.barplot(data=device_summary,x='device',y='CTR',hue='group')
plt.title('CTR by Channel')
plt.show()

#不同设备CVR对比图
plt.figure(figsize=(10,5))
sns.barplot(data=device_summary,x='device',y='CVR',hue='group')
plt.title('CVR by Channel')
plt.show()

    device group  users  clicks  orders       CTR       CVR
0  android     A  49894    4010     162  0.080370  0.040399
1  android     B  50318    4833     283  0.096049  0.058556
2      ios     A  34926    2830     128  0.081028  0.045230
3      ios     B  34909    3417     195  0.097883  0.057068
4      web     A  14948    1241      79  0.083021  0.063658


<Figure size 720x360 with 1 Axes>

<Figure size 720x360 with 1 Axes>

<p>新版首页在Android、iOS和Web设备上均提升了CTR与CVR，说明首页改版具有较好的普适性。</p><p></p><p style="text-align: left;">Web端CTR提升最明显，说明新版页面布局能够更有效吸引Web用户点击。</p><p style="text-align: left;"></p><p style="text-align: left;">Android端CVR提升最大，说明新版首页能够更有效促进移动端用户完成购买转化。</p><p style="text-align: left;"></p><p style="text-align: left;">Web端CVR始终保持最高水平，表明Web用户具有更强的购买意图。</p><p style="text-align: left;"></p><p style="text-align: left;">建议： 优先扩大Android端新版首页流量覆盖； 持续优化Web端商品推荐策略； 进一步分析不同设备用户的浏览路径和转化漏斗</p>

<p>为什么Web CTR最高，同时CVR也最高？</p><p style="text-align: left;">我认为Web端用户通常具有更强的购物目的性，例如搜索商品或进行价格比较，因此点击行为和购买行为都更积极。同时Web端页面展示空间更大，新版首页的推荐模块和视觉优化能够得到更充分展示，因此CTR和CVR均表现较好。</p>

<h2>6.项目总结</h2><p></p>

<p><strong>核心发现</strong>：</p><p style="text-align: left;">新版首页CTR和CVR提升10.03%和31.54%，说明新版首页能够有效吸引用户点击商品，同时提高购买转化率，且CTR与CVR的Z-Test结果均达到统计显著水平，因此结果可信。</p><p style="text-align: left;"></p><p style="text-align: left;"><strong>用户洞察</strong>：</p><p style="text-align: left;">新老用户：新版首页对新用户提升更明显，说明新版页面能够帮助首次访问用户更快发现感兴趣内容。</p><p style="text-align: left;"></p><p style="text-align: left;">渠道分析：所有渠道CTR均出现提升。其中Direct渠道CVR提升最大，说明高意图用户受益最明显。</p><p style="text-align: left;"></p><p style="text-align: left;">设备分析：新版首页在所有设备均有效，Web端CTR最高，Android端CVR提升最大，说明新版页面兼具吸引力与转化能力。</p><p style="text-align: left;"></p><p style="text-align: left;"><strong>业务建议</strong>：</p><p style="text-align: left;">逐步扩大新版首页流量占比。</p><p style="text-align: left;"></p><p style="text-align: left;">优先覆盖新用户流量。</p><p style="text-align: left;"></p><p style="text-align: left;">优先在Direct渠道推广新版首页。</p><p style="text-align: left;"></p><p style="text-align: left;">持续监控GMV和长期留存表现。</p><p style="text-align: left;"></p><p style="text-align: left;">电商平台首页改版A/B Test增长实</p>